[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Files, Paths and Formats](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)

# Writing Safely


## What you will be able to do

Replace a file in a way that survives the program being interrupted partway through, so the
original is either fully replaced or left exactly as it was.


## The idea

### The problem

The **Reading and Writing Text** notebook showed that opening a file with `"w"` empties it
immediately, before any of your code runs. That is fine when the write completes.

Consider what happens when it does not.

A script reads a data file, computes something, and writes the result back over the original. It
gets halfway through and the machine loses power, or the process is killed, or the code raises
on row 40,000 of 50,000. The file that was there is gone, and what has replaced it is a fragment
that stops mid-record.

The data is not corrupted in any interesting way. It is simply half of it, in a file with the
right name and the right modification time, and nothing on disk says so.

This is not rare. It is what happens every time a long-running job is interrupted, and it is
worth solving once.

### The pattern

> Write the new content to a **temporary file** beside the target. When it is complete and
> correct, **replace** the target with it in a single operation.
>
> `os.replace(temporary, target)` performs that swap. Either the target is the old file or it is
> the new one, and there is no moment where it is neither.

That single operation is the whole idea. The filesystem guarantees it, which is why the swap
must be a rename rather than a copy.

Three consequences follow, and each matters.

**The temporary file must be on the same filesystem** as the target, which in practice means in
the same folder. A rename across filesystems is a copy followed by a delete, and a copy can be
interrupted.

**Use `os.replace`, not `os.rename`.** `os.replace` overwrites an existing target on every
platform. `os.rename` does so on Unix and raises `FileExistsError` on Windows, which is a bug
you will not see until someone else runs your code.

**Nothing is lost if you crash before the replace.** The original is untouched and the
temporary file is left behind, which is why the cleanup belongs in a `finally`.

### Where you will meet this

Any script that overwrites a file it also reads. Any long-running job that writes results. Any
configuration file a program updates.

The **A Small Pipeline** notebook, which closes this guide, writes its output this way.

### What this notebook covers

- An interrupted write, run twice: once naively and once safely
- `tempfile`, and why the temporary file goes in the target's folder
- `os.replace`, and how it differs from `os.rename` and from `shutil.move`
- Validating the new content before it replaces anything
- `flush` and `fsync`, and what they actually promise
- `TemporaryDirectory`, for work that needs a whole folder
- Keeping a backup, and when that is the better answer
- Three errors, plus a write that succeeds and leaves the wrong thing on disk

### A first look

Nothing to run yet.

```python
import os, tempfile

fd, tmp = tempfile.mkstemp(dir=target.parent)

with os.fdopen(fd, "w", encoding="utf-8") as f:
    f.write(new_content)

os.replace(tmp, target)
```

Four lines. If anything raises before the last one, `target` still holds what it held before.


## Setup

Five imports and a folder to work in.

- `os` provides `os.replace`, `os.fdopen` and `os.fsync`
- `tempfile` creates the temporary file safely, without a name that could collide
- `json` supplies content that is either valid or obviously not, for the validation section
- `Path` builds paths and reads files back
- `shutil` copies a backup, and removes the scratch folder at the end

**Run this cell before the rest of the notebook.**


In [1]:

import os
import tempfile
import json
from pathlib import Path
import shutil

scratch = Path("scratch")
scratch.mkdir(exist_ok=True)


def fresh_data_file():
    """Recreate the file each demonstration starts from."""
    target = scratch / "data.json"
    target.write_text(
        json.dumps({"records": [0, 1, 2, 3, 4], "status": "complete"}),
        encoding="utf-8",
    )
    return target


print(fresh_data_file().read_text(encoding="utf-8"))


{"records": [0, 1, 2, 3, 4], "status": "complete"}


## Worked examples

### The naive write, interrupted

This is what most code does, and what happens when it does not finish.


In [2]:

target = fresh_data_file()

print("before:", target.read_text(encoding="utf-8"))

try:
    with open(target, "w", encoding="utf-8") as f:
        f.write('{"records": [0, 1, 2')
        raise RuntimeError("interrupted here")
except RuntimeError as e:
    print("failed: ", e)

print("after: ", repr(target.read_text(encoding="utf-8")))


before: {"records": [0, 1, 2, 3, 4], "status": "complete"}
failed:  interrupted here
after:  '{"records": [0, 1, 2'


The original is gone. What is left is a fragment, in a file with the correct name.

Note that the fragment exists at all only because `with` closed the file on the way out, which
flushed what had been written. A process killed outright would have left the file empty
instead. Both outcomes are the same problem.

And the file is not merely incomplete. It is no longer readable:


In [3]:

try:
    json.loads(target.read_text(encoding="utf-8"))
except json.JSONDecodeError as e:
    print("JSONDecodeError:", e.msg, "at column", e.colno)


JSONDecodeError: Expecting ',' delimiter at column 21


### The same interruption, done safely


In [4]:

target = fresh_data_file()

print("before:", target.read_text(encoding="utf-8"))

temporary = None
try:
    handle, temporary = tempfile.mkstemp(dir=target.parent, prefix="data", suffix=".tmp")
    with os.fdopen(handle, "w", encoding="utf-8") as f:
        f.write('{"records": [0, 1, 2')
        raise RuntimeError("interrupted here")
    os.replace(temporary, target)
except RuntimeError as e:
    print("failed: ", e)
finally:
    if temporary is not None:
        Path(temporary).unlink(missing_ok=True)

print("after: ", target.read_text(encoding="utf-8"))


before: {"records": [0, 1, 2, 3, 4], "status": "complete"}
failed:  interrupted here
after:  {"records": [0, 1, 2, 3, 4], "status": "complete"}


Identical failure, and the original is exactly as it was. The half-written content went into the
temporary file, which the `finally` removed.

`os.replace` was never reached, and that is the point: the target only changes on the line that
succeeds atomically.

Now the same code, allowed to finish:


In [5]:

target = fresh_data_file()

handle, temporary = tempfile.mkstemp(dir=target.parent, prefix="data", suffix=".tmp")
with os.fdopen(handle, "w", encoding="utf-8") as f:
    f.write(json.dumps({"records": list(range(10)), "status": "complete"}))

os.replace(temporary, target)

print("after: ", target.read_text(encoding="utf-8"))
print("temporary file gone:", not Path(temporary).exists())


after:  {"records": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], "status": "complete"}
temporary file gone: True


The temporary file is gone because `os.replace` moved it rather than copying it. There is no
cleanup to do on the success path.

### Why mkstemp and os.fdopen

`tempfile.mkstemp` returns two things: an already-open low-level file handle, and the path it
created. `os.fdopen` turns that handle into the ordinary file object you know.

The reason for the dance is that `mkstemp` creates the file **and** opens it in one step, so
there is no window in which another program could create a file with the same name. Generating
a name yourself and then opening it has that window, and it is the kind of bug that appears only
under load.

`dir=target.parent` is the important argument, and it is why the temporary file appears beside
the target rather than in the system temporary folder.


In [6]:

target = fresh_data_file()

handle, temporary = tempfile.mkstemp(dir=target.parent, prefix="data", suffix=".tmp")
os.close(handle)

print("target name:   ", target.name)
print("temporary name:", Path(temporary).name)
print("same folder:   ", Path(temporary).parent == target.parent.resolve()
      or Path(temporary).parent == target.parent)

Path(temporary).unlink()


target name:    data.json
temporary name: datanfe619cu.tmp
same folder:    True


Same folder means same filesystem, and same filesystem is what makes the replace a single
operation rather than a copy.

Put the temporary file in `/tmp` and the replace becomes a copy across devices, which can be
interrupted, which is the problem you were solving.


### replace, rename and move

Three functions that look interchangeable and are not.

| Call | Overwrites an existing target | Atomic |
|---|---|---|
| `os.replace(src, dst)` | yes, on every platform | yes, same filesystem |
| `os.rename(src, dst)` | on Unix yes, on Windows raises | yes, same filesystem |
| `shutil.move(src, dst)` | yes | no, may copy |

`os.replace` is the one to use.


In [7]:

first = scratch / "first.txt"
second = scratch / "second.txt"
first.write_text("content of first", encoding="utf-8")
second.write_text("content of second", encoding="utf-8")

os.replace(first, second)

print("second now holds:", second.read_text(encoding="utf-8"))
print("first still exists:", first.exists())


second now holds: content of first
first still exists: False


`os.rename` would have done the same thing here, because this notebook is not running on
Windows. On Windows the same line raises `FileExistsError`, so code written and tested on a Mac
fails the first time a colleague runs it.

`shutil.move` handles crossing filesystems by copying, which is useful and is exactly the
property you do not want here.


### Check before you replace

The pattern gets better with one addition: read the temporary file back and confirm it is what
you meant before swapping it in.


In [8]:

def write_checked(path, text, check):
    """Write text to path, replacing it only if check() accepts the result."""
    handle, temporary = tempfile.mkstemp(dir=path.parent, prefix=path.name, suffix=".tmp")
    try:
        with os.fdopen(handle, "w", encoding="utf-8") as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())
        check(Path(temporary))
        os.replace(temporary, path)
        return True
    except Exception as e:
        Path(temporary).unlink(missing_ok=True)
        print(f"  refused: {type(e).__name__}: {e}")
        return False


def must_be_json(path):
    json.loads(path.read_text(encoding="utf-8"))


config = scratch / "config.json"
config.write_text('{"version": 1}', encoding="utf-8")

print("valid content: ", write_checked(config, '{"version": 2}', must_be_json))
print("file now:      ", config.read_text(encoding="utf-8"))


valid content:  True
file now:       {"version": 2}


In [9]:

print("broken content:", write_checked(config, '{"version": ', must_be_json))
print("file still:    ", config.read_text(encoding="utf-8"))


  refused: JSONDecodeError: Expecting value: line 1 column 13 (char 12)
broken content: False
file still:     {"version": 2}


The broken write was refused, the temporary file was removed, and the previous version is still
there.

That check is worth more than it looks. Writing a truncated file is a rare accident; writing a
**complete** file with wrong content is a common one, and validating the result catches both.


### flush and fsync

`f.flush()` pushes Python's buffer to the operating system. `os.fsync()` asks the operating
system to push its own buffer to the physical disk.

Without the second, a completed write can still be lost if the machine loses power in the next
moment, because the data was in the operating system's cache rather than on the disk.

It is worth including for anything that matters, and it is not free: `fsync` waits for the disk.
Calling it once per file is sensible, and calling it in a loop over ten thousand small files is
how a script becomes a hundred times slower.


### TemporaryDirectory

When the work needs a whole folder rather than one file, `TemporaryDirectory` creates one and
removes it on the way out, including when the block raises.


In [10]:

with tempfile.TemporaryDirectory() as folder:
    workspace = Path(folder)
    (workspace / "part1.txt").write_text("first", encoding="utf-8")
    (workspace / "part2.txt").write_text("second", encoding="utf-8")
    print("inside: ", sorted(p.name for p in workspace.iterdir()))

print("after:  exists =", workspace.exists())


inside:  ['part1.txt', 'part2.txt']
after:  exists = False


This is the `with` statement from the **Reading and Writing Text** notebook doing the same job
for a directory. Everything inside is gone when the block ends, so an interrupted run leaves
nothing behind to clean up later.


### Keeping a backup

Atomic replacement guarantees you never see a half-written file. It does not let you go back to
the previous version.


In [11]:

important = scratch / "important.txt"
important.write_text("version 1", encoding="utf-8")

backup = important.with_suffix(important.suffix + ".bak")
shutil.copy2(important, backup)

important.write_text("version 2", encoding="utf-8")

print("current:", important.read_text(encoding="utf-8"))
print("backup: ", backup.read_text(encoding="utf-8"))


current: version 2
backup:  version 1


`copy2` copies the modification time along with the contents, which makes the backup look like
what it is rather than like a file created just now.

Use a backup when the previous version has value and the new one might be wrong. Use atomic
replacement when what matters is that the file is never half-written. They solve different
problems and combine well.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/09-writing-safely-solutions.ipynb).

**1.** Write `scratch/notes.txt` with three lines, then overwrite it naively and raise partway
through. Print what is left.


In [12]:
# your code here


**2.** Do the same thing with a temporary file and `os.replace`, and show the original survived.


In [13]:
# your code here


**3.** Now let the safe version complete, and confirm both the new content and that no temporary
file remains.


In [14]:
# your code here


**4.** Write a function `safe_write(path, text)` that does the temporary-file dance, cleaning up
in a `finally`. Use it once.


In [15]:
# your code here


**5.** Extend it to refuse content that is not valid JSON, and show it refusing.


In [16]:
# your code here


**6.** Use `TemporaryDirectory` to create two files, print them, and show the folder is gone
afterward.


In [17]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### FileNotFoundError: the temporary file was already moved


In [18]:
target = fresh_data_file()

handle, temporary = tempfile.mkstemp(dir=target.parent, suffix=".tmp")
os.close(handle)

os.replace(temporary, target)

try:
    Path(temporary).unlink()
except FileNotFoundError as e:
    print(type(e).__name__, "-", e.strerror)
    print("it looked for:", Path(temporary).name)


FileNotFoundError - No such file or directory
it looked for: tmpbswq5ks9.tmp


`os.replace` **moved** the temporary file, so deleting it afterward fails: there is nothing at
that name any more. The exception is caught here rather than raised, because its message
contains the full path of the temporary file, which differs on every machine.

This is why the cleanup belongs in a `finally` with `missing_ok=True`, or why the success path
should not clean up at all.


In [19]:

Path(temporary).unlink(missing_ok=True)

print("missing_ok makes it a no-op")


missing_ok makes it a no-op


### OSError: replacing across filesystems

`os.replace` is atomic only within one filesystem. Across two, the behavior depends on the
platform, and on Linux it raises.


In [20]:

handle, elsewhere = tempfile.mkstemp()      # no dir= given, so the system temporary folder
os.close(handle)

print("temporary is in:", "/".join(Path(elsewhere).parts[1:3]))
print("target is under:", scratch)
print("same filesystem here, so this would work on this machine")

Path(elsewhere).unlink()


temporary is in: var/folders
target is under: scratch
same filesystem here, so this would work on this machine


On this machine both locations happen to be on one filesystem, so the replace would succeed. On
a machine where `/tmp` is a separate filesystem, the same code raises
`OSError: Invalid cross-device link`.

Passing `dir=target.parent` removes the question entirely, which is why every example in this
notebook does.


### PermissionError: replacing a file that is open

On Windows, replacing a file another program has open fails. On Unix it usually succeeds, and
the other program keeps reading the old contents until it closes the file.

That difference is worth knowing rather than demonstrating, because the failure only appears on
one platform, and this notebook is not running on it.


### The quiet one: the write succeeded and the content is wrong


In [21]:

target = fresh_data_file()

records = {"records": [0, 1, 2], "status": "complete"}

handle, temporary = tempfile.mkstemp(dir=target.parent, suffix=".tmp")
with os.fdopen(handle, "w", encoding="utf-8") as f:
    f.write(str(records))          # str(), not json.dumps()

os.replace(temporary, target)

print("file now:", target.read_text(encoding="utf-8"))


file now: {'records': [0, 1, 2], 'status': 'complete'}


No error at any stage. The temporary file was written, the replace succeeded, and the original
is gone.

The content is Python's `repr` rather than JSON: single quotes instead of double. It will not
load, and nothing in the writing process could have noticed.


In [22]:

try:
    json.loads(target.read_text(encoding="utf-8"))
except json.JSONDecodeError as e:
    print("JSONDecodeError:", e.msg)


JSONDecodeError: Expecting property name enclosed in double quotes


Atomic replacement protects against **interruption**, not against being wrong. That is what the
validation step is for, and it is the reason `write_checked` reads the file back rather than
trusting the string it was handed.


### Cleaning up


In [23]:

shutil.rmtree(scratch)

print("scratch still there:", scratch.exists())


scratch still there: False


## Recap

- Opening a file with `"w"` destroys it immediately, so an interrupted write leaves nothing.
- Write to a temporary file, then `os.replace` it onto the target in one operation.
- Put the temporary file in the target's folder, so the replace stays on one filesystem.
- `os.replace` overwrites on every platform; `os.rename` raises on Windows; `shutil.move` may
  copy.
- `tempfile.mkstemp` creates and opens in one step, so no other program can claim the name.
- Clean up the temporary file in a `finally`, with `missing_ok=True`.
- Read the new file back and validate it before replacing anything.
- `flush` reaches the operating system; `fsync` reaches the disk, and costs time.
- `TemporaryDirectory` does the same job for a whole folder.
- Atomic replacement prevents a half-written file. It does not prevent a wrong one.


## What is next

The **A Small Pipeline** notebook, the last in this guide. It puts every notebook here together
on one realistic task: read many files of mixed quality, clean them, report what was dropped,
and write a single result safely.


---

&#8592; **Previous:** [Compression and Archives](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/08-compression-and-archives.ipynb)  &nbsp;·&nbsp;  [Files, Paths and Formats Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/files-paths-and-formats.html)  &nbsp;·&nbsp;  **Next:** [A Small Pipeline](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/files-paths-and-formats/10-a-small-pipeline.ipynb) &#8594;
